# Build Vision agents by using Smolagents

### Install the dependencies

In [ ]:
! pip install smolagents[telemetry] opentelemetry-sdk opentelemetry-exporter-otlp openinference-instrumentation-smolagents
! pip install smolagents -U duckduckgo-search ddgs huggingface_hub

INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opentelemetry-instrumentation to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of pydantic-ai-slim[ag-ui,anthropic,bedrock,cli,cohere,evals,fastmcp,google,groq,huggingface,logfire,mcp,mistral,openai,retries,spec,temporal,ui,vertexai,xai] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of pydantic-ai-slim[ag-ui,anthropic,bedrock,cli,cohere,evals,fastmcp,google,groq,huggingface,logfire,mcp,mistral,openai,retries,spec,temporal,ui,vertexai,xai] to determine which version is compatible with other requirements. This could take a while.
INFO: This

### Login to your Hugging Face account to access the Inference API

In [ ]:
from huggingface_hub import login
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN') # Access the secret
if HF_TOKEN:
    login(HF_TOKEN)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token not found or notebook access not enabled.")

Successfully logged in to Hugging Face!


### Load the images to process

In [ ]:
from PIL import Image
import requests
from io import BytesIO

image_urls = [
    "https://upload.wikimedia.org/wikipedia/commons/e/e8/The_Joker_at_Wax_Museum_Plus.jpg",
    "https://upload.wikimedia.org/wikipedia/en/9/98/Joker_%28DC_Comics_character%29.jpg"
]

images = []
for url in image_urls:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
    }
    response = requests.get(url,headers=headers)
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images.append(image)

### Set OpenAI API Key

In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = ""

### Set Langfuse public and secret keys; And, connect to Langfuse

In [ ]:
import base64
import os

os.environ["LANGFUSE_PUBLIC_KEY"] = ""
os.environ["LANGFUSE_SECRET_KEY"] = ""

LANGFUSE_PUBLIC_KEY = os.environ.get("LANGFUSE_PUBLIC_KEY")
LANGFUSE_SECRET_KEY = os.environ.get("LANGFUSE_SECRET_KEY")

if not LANGFUSE_PUBLIC_KEY or not LANGFUSE_SECRET_KEY:
    raise ValueError("Langfuse public or secret keys are missing!")

LANGFUSE_AUTH = base64.b64encode(f"{LANGFUSE_PUBLIC_KEY}:{LANGFUSE_SECRET_KEY}".encode()).decode()

#os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://cloud.langfuse.com/api/public/otel" # EU data region
os.environ["OTEL_EXPORTER_OTLP_ENDPOINT"] = "https://us.cloud.langfuse.com/api/public/otel" # US data region
os.environ["OTEL_EXPORTER_OTLP_HEADERS"] = f"Authorization=Basic {LANGFUSE_AUTH}"

print("Langfuse environment variables set successfully!")


Langfuse environment variables set successfully!


In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from openinference.instrumentation.smolagents import SmolagentsInstrumentor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import SimpleSpanProcessor

trace_provider = TracerProvider()
trace_provider.add_span_processor(SimpleSpanProcessor(OTLPSpanExporter()))

SmolagentsInstrumentor().instrument(tracer_provider=trace_provider)

### Initialize the gpt-4o model; And, run the agent by passing the images to analyze

In [ ]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(model_id="gpt-4o")

# Instantiate the agent
agent = CodeAgent(
    tools=[],
    model=model,
    max_steps=20,
    verbosity_level=2
)

response = agent.run(
    """
    Describe the costume and makeup that the comic character in these photos is wearing and return the description.
    Tell me if the guest is The Joker or Wonder Woman.
    """,
    images=images
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Describe the costume and makeup that the comic character in these photos is wearing and return the description. │
│     Tell me if the guest is The Joker or Wonder Woman.                                                          │
│                                                                                                                 │
╰─ OpenAIModel - gpt-4o ──────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
The character depicted in the images is The Joker.                                                                 
                                                                                                                   
### Description:                                                                                                   
                                                                                                                   
- **Costume:**                                                                                                     
  - A vibrant purple suit.                                                                                         
  - A yellow or orange shirt underneath.                                                                           
  - A large purple tie, fitting the eccentric style of The Joker.                                                  
                                                                                                                   
- **Makeup:**                                                                                                      
  - White face paint covering the entire face.                                                                     
  - High-arched eyebrows with dark lines emphasizing the expression.                                               
  - Blue eye shadow.                                                                                               
  - Red lips with an exaggerated grin.                                                                             
                                                                                                                   
The iconic purple suit and the distinctive makeup are classic features of The Joker's appearance.                  

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
            Here is your code snippet:
            The character depicted in the images is The Joker.

### Description:

- **Costume:**
  - A vibrant purple suit.
  - A yellow or orange shirt underneath.
  - A large purple tie, fitting the eccentric style of The Joker.

- **Makeup:**
  - White face paint covering the entire face.
  - High-arched eyebrows with dark lines emphasizing the expression.
  - Blue eye shadow.
  - Red lips with an exaggerated grin.

The iconic purple suit and the distinctive makeup are classic features of The Joker's appearance.</code>
            Make sure to include code with the correct pattern, for instance:
            Thoughts: Your thoughts
            <code>
            # Your python code here
            </code>
Make sure to provide correct code blobs.

[Step 1: Duration 5.84 seconds| Input tokens: 3,023 | Output tokens: 112]

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I need to describe the costume and makeup of the character in the images and identify if it's The Joker or
Wonder Woman.                                                                                                      
                                                                                                                   
<code>                                                                                                             
# Description of the character                                                                                     
                                                                                                                   
guest = "The Joker"                                                                                                
                                                                                                                   
costume_description = {                                                                                            
    "costume": [                                                                                                   
        "Vibrant purple suit",                                                                                     
        "Yellow or orange shirt",                                                                                  
        "Large purple tie"                                                                                         
    ],                                                                                                             
    "makeup": [                                                                                                    
        "White face paint covering entire face",                                                                   
        "High-arched eyebrows",                                                                                    
        "Blue eye shadow",                                                                                         
        "Red lips with exaggerated grin"                                                                           
    ]                                                                                                              
}                                                                                                                  
                                                                                                                   
# Return the detailed description and character identity                                                           
final_answer(f"The guest is {guest}. Costume: {costume_description['costume']}. Makeup:                            
{costume_description['makeup']}.")                                                                                 
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Description of the character                                                                                   
                                                                                                                   
  guest = "The Joker"                                                                                              
                                                                                                                   
  costume_description = {                                                                                          
      "costume": [                                                                                                 
          "Vibrant purple suit",                                                                                   
          "Yellow or orange shirt",                                                                                
          "Large purple tie"                                                                                       
      ],                                                                                                           
      "makeup": [                                                                                                  
          "White face paint covering entire face",                                                                 
          "High-arched eyebrows",                                                                                  
          "Blue eye shadow",                                                                                       
          "Red lips with exaggerated grin"                                                                         
      ]                                                                                                            
  }                                                                                                                
                                                                                                                   
  # Return the detailed description and character identity                                                         
  final_answer(f"The guest is {guest}. Costume: {costume_description['costume']}. Makeup:                          
  {costume_description['makeup']}.")                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: The guest is The Joker. Costume: ['Vibrant purple suit', 'Yellow or orange shirt', 'Large purple 
tie']. Makeup: ['White face paint covering entire face', 'High-arched eyebrows', 'Blue eye shadow', 'Red lips with 
exaggerated grin'].

[Step 3: Duration 5.16 seconds| Input tokens: 13,154 | Output tokens: 381]

### Print the response

In [ ]:
response

({'suit': 'A vibrant and distinctive purple suit.',
  'shirt': 'Bright yellow or orange shirt.',
  'tie': 'Purple tie or bow.',
  'makeup': 'White face makeup with exaggerated red lips.',
  'eyes': 'Dark, heavily outlined eyes with blue or green hues.',
  'hair': 'Green hair.'},
 'The Joker')